# Lecture 2, Advanced: fine-tuning on the glossary compared with glossary search

The course glossary (`data/glossary.json`, 58 terms) is converted into question-answer pairs with `scripts/make_sft_dataset.py`, and `Qwen/Qwen3.5-0.8B` is trained on them with LoRA. Six terms are left out of training. The same 26 test questions are then answered under four conditions: the base model and the LoRA model, each without and with the glossary search in `app/tools.py` (the search used by the lecture 3 agent).

| Group | What it tests |
|---|---|
| trained, training wording | recall of a trained definition, asked in the training wording |
| trained, new wording | recall when the question is reworded |
| trained, described without the term | recall when the term is not named, which a name-based search cannot match |
| in glossary, not trained | knowledge held only in the glossary |
| not in glossary | terms that do not exist; a correct answer declines |

An answer is counted as correct when it contains every listed phrase from the glossary definition or, for terms not in the glossary, when it declines. Executed on Apple silicon (MPS); the setup cells follow the course notebook. A first attempt with the course batch settings (4 per step, accumulation 2) diverged on MPS: the loss rose to 5.57 and then became 0, and the trained model produced only `!`. This run uses 1 example per step with accumulation 8, the same effective batch of 8 without padding; the first attempt is kept in `../results/advanced_glossary/result_first_attempt_diverged.json`.

In [1]:
import platform, torch
print(platform.machine(), platform.mac_ver()[0], '| MPS:', torch.backends.mps.is_available(), '| CUDA:', torch.cuda.is_available())


arm64 27.0.1 | MPS: True | CUDA: False


In [2]:
PINS = {"transformers": "5.18.0", "peft": "0.21.1", "trl": "1.14.2", "accelerate": "1.15.0", "datasets": "5.1.0"}
import transformers, peft, trl, accelerate, datasets
versions = {m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]}
print(versions)
assert all(versions[k] == v for k, v in PINS.items()), 'library versions differ from the pinned ones'


<site-packages>/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.11.0'}


## 1. Settings

In [3]:
from pathlib import Path
import json, subprocess, sys, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 8,
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 150,
    "gen": {"do_sample": False},
    "work_dir": "../results/advanced_glossary",
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)
REPO = Path("../../..")   # repository root, relative to this notebook
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))


{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 8,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 150,
  "gen": {
    "do_sample": false
  },
  "work_dir": "../results/advanced_glossary"
}


## 2. Training data and test questions

In [4]:
# Training data comes from the course script, unchanged: two question templates per glossary term
lines = subprocess.run([sys.executable, str(REPO / "scripts" / "make_sft_dataset.py")], capture_output=True, text=True, check=True).stdout.splitlines()
records = [json.loads(l) for l in lines]
glossary = json.loads((REPO / "data" / "glossary.json").read_text(encoding="utf-8"))["terms"]
assert len(records) == 2 * len(glossary)   # the script writes two records per term, in glossary order
by_id = {t["id"]: t for t in glossary}

# Six course-specific terms are trained and tested; six other terms stay in the glossary but are left out of training.
# Each list holds phrases from the glossary definition that a correct answer has to contain.
TRAINED = {
    "homework-levels": ["講義内容の確認", "どれか1つ"],
    "answer-md": ["lectures/NN/answer.md", "PR"],
    "lec-start-tag": ["Gitタグ", "lec03-start"],
    "default-model": ["gemini-3.5-flash-lite", "AIE_MODEL"],
    "codespaces-secret": ["環境変数", "再起動"],
    "test-model": ["偽モデル", "APIキー"],
}
HELD_OUT = {
    "harness": ["ループ", "read・write・edit・bash"],
    "pi": ["MITライセンス", "pi-coding-agent"],
    "tool-namespace": ["サーバー名_ツール名", "接頭辞"],
    "mcp-config": ["mcpServers", "app/mcp_servers.json"],
    "fetch-server": ["Markdown", "mcp-server-fetch"],
    "lora-rank": ["低ランク行列", "ハイパーパラメータ"],
}
for i, words in (TRAINED | HELD_OUT).items():
    assert all(w in by_id[i]["definition"] for w in words), i   # every test phrase comes from the definition

train_records = [r for n, r in enumerate(records) if glossary[n // 2]["id"] not in HELD_OUT]
(WORK / "glossary_sft.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in train_records), encoding="utf-8")

DESCRIBED = {   # trained terms, asked about without naming them
    "homework-levels": "AIE2026 の宿題は何段階に分かれていて、いくつ取り組めば完了ですか？",
    "default-model": "AIE2026 の演習で標準として使うモデルの名前は何ですか？",
    "lec-start-tag": "前の回が終わっていなくても、その回の演習を始められるように講座repoに付いている目印は何ですか？",
    "codespaces-secret": "Codespaces の中で API キーを環境変数として受け取るには、GitHub のどこに登録しますか？",
}
NOT_IN_GLOSSARY = ["lec99-final タグ", "AIE2026 の中間レポート", "講座ボーナスレベル", "OmniCampus ポイント"]
ABSTAIN_MARKERS = ["記載がありません", "見つかりません", "存在しません", "分かりません", "わかりません", "情報がありません", "確認が必要"]

tests = []
for i, words in TRAINED.items():
    term = by_id[i]["term"]
    tests.append({"group": "trained, training wording", "id": i, "question": f"「{term}」とは何ですか？", "must_include": words})
    tests.append({"group": "trained, new wording", "id": i, "question": f"AIE2026 の講座で使う「{term}」は、どういう意味ですか？", "must_include": words})
for i, q in DESCRIBED.items():
    tests.append({"group": "trained, described without the term", "id": i, "question": q, "must_include": TRAINED[i]})
for i, words in HELD_OUT.items():
    tests.append({"group": "in glossary, not trained", "id": i, "question": f"「{by_id[i]['term']}」とは何ですか？", "must_include": words})
for term in NOT_IN_GLOSSARY:
    tests.append({"group": "not in glossary", "id": None, "question": f"「{term}」とは何ですか？", "must_include": None})
GROUPS = list(dict.fromkeys(t["group"] for t in tests))
print(f"train={len(train_records)} records ({len(glossary) - len(HELD_OUT)} terms x 2 templates) | test={len(tests)}")
print({g: sum(t["group"] == g for t in tests) for g in GROUPS})


train=104 records (52 terms x 2 templates) | test=26
{'trained, training wording': 6, 'trained, new wording': 6, 'trained, described without the term': 4, 'in glossary, not trained': 6, 'not in glossary': 4}


## 3. Model, prompts and the base model's answers

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
sys.path.insert(0, str(REPO))
from app.tools import search_glossary   # the glossary search used by the lecture 3 agent

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
print("loaded:", CONFIG["base_model"], "| device:", DEVICE)

RAG_SYSTEM = "あなたは講座の用語集の回答係です。用語集の検索結果だけを根拠に答えてください。検索結果に該当する用語が無い場合は「用語集に記載がありません」と答えてください。"

def closed_book(question):
    # the same shape as the training records: one user turn, no system prompt
    return [{"role": "user", "content": question}]

def with_search(question):
    return [{"role": "system", "content": RAG_SYSTEM},
            {"role": "user", "content": f"用語集の検索結果:\n{search_glossary(question)}\n\n質問: {question}"}]

def generate(model, messages):
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


loaded: Qwen/Qwen3.5-0.8B | device: mps


In [6]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

def squash(text):
    # the model writes "API キー" where the glossary has "APIキー"; compare without spaces, as app/tools.py does
    return text.replace(" ", "").replace("　", "")

def score(test, answer):
    if test["must_include"] is None:   # not in the glossary: correct only if the answer declines
        return any(m in squash(answer) for m in ABSTAIN_MARKERS)
    return all(squash(w) in squash(answer) for w in test["must_include"])

def evaluate(model, condition, prompt_fn):
    rows, t0 = [], time.time()
    for t in tests:
        answer = generate(model, prompt_fn(t["question"]))
        rows.append({"condition": condition, **t, "answer": answer, "correct": score(t, answer)})
    print(f"{condition}: {sum(r['correct'] for r in rows)}/{len(rows)} correct, {time.time() - t0:.0f} s")
    return rows

rows = evaluate(model, "base, closed book", closed_book)
rows += evaluate(model, "base, with search", with_search)


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


base, closed book: 0/26 correct, 99 s


base, with search: 14/26 correct, 50 s


## 4. LoRA training on the glossary records

In [7]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

train_ds = Dataset.from_list(train_records)   # already in the messages format
model.config.use_cache = False
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=1, gradient_accumulation_steps=8,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_strategy="no",
                max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
t0 = time.time()
trainer.train()
TRAIN_SECONDS = round(time.time() - t0)
LOSSES = [round(h["loss"], 4) for h in trainer.state.log_history if "loss" in h]
print(f"training time: {TRAIN_SECONDS} s | steps: {trainer.state.global_step} | epochs: {trainer.state.epoch:.1f}")
print("training loss every 10 steps:", LOSSES)
import math
assert all(math.isfinite(l) and l > 0 for l in LOSSES), "training diverged (loss became 0 or not finite)"
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()


Tokenizing train dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

Tokenizing train dataset: 100%|██████████| 104/104 [00:00<00:00, 2607.96 examples/s]

Building labels for train dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

Building labels for train dataset: 100%|██████████| 104/104 [00:00<00:00, 15293.72 examples/s]

Truncating train dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

...


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


<site-packages>/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss
10,3.298102
20,2.107987
30,1.391762
40,0.832041
50,0.410317
60,0.224530
70,0.145654
80,0.116519
90,0.104873
100,0.096285


training time: 2378 s | steps: 104 | epochs: 8.0
training loss every 10 steps: [3.2981, 2.108, 1.3918, 0.832, 0.4103, 0.2245, 0.1457, 0.1165, 0.1049, 0.0963]


## 5. The LoRA model's answers and the comparison

In [8]:
rows += evaluate(tuned, "LoRA, closed book", closed_book)
rows += evaluate(tuned, "LoRA, with search", with_search)
CONDITIONS = list(dict.fromkeys(r["condition"] for r in rows))
summary = (pd.DataFrame(rows).groupby(["group", "condition"], sort=False)["correct"]
           .agg(lambda s: f"{int(s.sum())}/{len(s)}").unstack()[CONDITIONS].loc[GROUPS])
display(summary)
display(pd.DataFrame(rows)[["condition", "group", "question", "answer", "correct"]])


LoRA, closed book: 8/26 correct, 47 s


LoRA, with search: 7/26 correct, 48 s


condition,"base, closed book","base, with search","LoRA, closed book","LoRA, with search"
group,,,,
"trained, training wording",0/6,4/6,6/6,4/6
"trained, new wording",0/6,5/6,2/6,2/6
"trained, described without the term",0/4,0/4,0/4,0/4
"in glossary, not trained",0/6,2/6,0/6,1/6
not in glossary,0/4,3/4,0/4,0/4


,condition,group,question,answer,correct
0,"base, closed book","trained, training wording",「宿題の3レベル」とは何ですか？,「宿題の 3 レベル」とは、**学習の進捗やレベルを 3 つの段階に分けて示す方法**です。これは、単一の目標を達成するだけでなく、段階的に進めていくプロセスや、達成した成果を評価する基準として使われることが多く、特に**「学習の進捗管理」**や**「学習成果の評価」**において非常に有用です。\n\n一般的に、この 3 段階は以下のようになります。\n\n### 1. 基本レベル（基礎的な学習）\nこのレベルは、学習の**基礎知識**や**基本概念**を掌握する段階です。\n* **内容**: 基本的な定義、公式、定理、用語の,False
1,"base, closed book","trained, new wording",AIE2026 の講座で使う「宿題の3レベル」は、どういう意味ですか？,AIE2026（アイルランド・イギリス・アメリカ・ヨーロッパ・アジア・オーストラリア・アジア太平洋地域）の講座で「宿題の 3 レベル」という表現は、**「課題の難易度や範囲」を 3 つの段階に分けて示すための標準的な分類法**です。\n\nこの 3 つのレベルは、単に「難しい」「簡単な」だけでなく、**「どのレベルで解くべきか」**を明確に定義しているため、学習者や指導者が目標を達成するための指針になります。\n\n具体的には、以下の 3 つのレベルに分けられます。\n\n### 1. 基礎レベル（Easy / Beginner）\n*,False
2,"base, closed book","trained, training wording",「answer.md」とは何ですか？,**`answer.md`** は、**AI（人工知能）の回答生成や回答の検証**において非常に重要なファイルです。\n\nこのファイルは、AI が特定の質問やタスクに対して回答を生成した際に、その回答が**正しいか、適切か、または誤解を招く可能性があるか**を自動でチェックするものです。\n\n主な機能と目的は以下の通りです：\n\n1. **回答の検証（Verification）**\n AI が生成した回答に対して、人間が確認できるかどうかを判断します。例えば、AI が「この問題を解く方法」を提案し、人間がそれを確認し、実際に問題に答えるかどうかを確認します。\n\n2. **誤解,False
3,"base, closed book","trained, new wording",AIE2026 の講座で使う「answer.md」は、どういう意味ですか？,AIE2026 の講座で使う `answer.md` というファイルは、**「回答（出力）ファイル」**という意味です。\n\n具体的には、講義の回答や課題の解答を、このファイルに保存して提出する際に使います。\n\n主な用途は以下の通りです：\n\n1. **講義の回答**\n * 講義の質問に対して、論理的な回答や論理的な思考プロセスを記述する。\n * 特定の論理の証明や、特定の論理の推論を記述する。\n\n2. **課題の解答**\n * 課題（Problem）に対して、解法やコードの出力結果を,False
4,"base, closed book","trained, training wording",「lecNN-start タグ」とは何ですか？,「lecNN-start」は、**LeetCode（リテール・コード・ネスト）の「LeetCode」のスタートアッププロジェクト**です。\n\nこのプロジェクトは、LeetCode の開発者コミュニティ（GitHub）を運営する「LeetCode 株式会社」が、2024 年 1 月 1 日に設立したものです。主な目的は、LeetCode の開発者コミュニティをより広く、かつ質の高いコミュニティに引き込むことで、LeetCode の発展と活発な開発活動を促進することです。\n\nこのプロジェクトは、以下の目的で活動しています：\n* LeetCode の開発者コミュニティの拡大と活性化\n* LeetCode の開発者コミュニティの質を高める,False
...,...,...,...,...,...
99,"LoRA, with search","in glossary, not trained",「ランク（LoRAのr）」とは何ですか？,LoRAで使う低ランク行列の大きさを決めるハイパーパラメータ。大きいほどモデルの表現力が上がるが、学習するパラメータ数とメモリも増える。,True
100,"LoRA, with search",not in glossary,「lec99-final タグ」とは何ですか？,lec99の最終回演習の集大成。演習の進め方、宿題の作成・提出、宿題の振り返り、演習の評価と結果の解釈、討論の組織と進め方。本講座の核心。,False
101,"LoRA, with search",not in glossary,「AIE2026 の中間レポート」とは何ですか？,講義の3回の演習（宿題の解答、宿題の修正、宿題の提出）を回す結果の集まり。回数を3に数えてレポートとして提出する。第3回の提出は提出日で終わるため、提出前の2回の修習を回す必要がある。提出する回数を教えてもらい、その中でレポートを作る。提出はオンラインのレポートシステムで、システムが確認する。確認は質問で、確認が通则に合うか、確認の結果（通りか失敗か）で決定する。確認の結果が通则に合うときはレポートとして提出し、確認が失敗したり、質問が返らないときはレポートを提出しない。,False
102,"LoRA, with search",not in glossary,「講座ボーナスレベル」とは何ですか？,用語集に該当しません。講座の検索結果にはない。講座の解説ページや配布資料でよく見られる。,False


## 6. Results file

In [9]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k != "work_dir"},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "device": DEVICE,
    "method": "LoRA (no quantisation) on glossary Q&A from scripts/make_sft_dataset.py",
    "train_records": len(train_records),
    "held_out_terms": list(HELD_OUT),
    "training_seconds": TRAIN_SECONDS,
    "training_loss": LOSSES,
    "group_sizes": {g: sum(t["group"] == g for t in tests) for g in GROUPS},
    "correct": {c: {g: sum(r["correct"] for r in rows if r["condition"] == c and r["group"] == g) for g in GROUPS} for c in CONDITIONS},
    "executed_training_myself": True,
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
(WORK / "eval_rows.json").write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "training_loss"}, ensure_ascii=False, indent=2))


{
  "config": {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 8,
    "warmup_steps": 6,
    "lr": 0.0002,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 150,
    "gen": {
      "do_sample": false
    }
  },
  "versions": {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "torch": "2.11.0"
  },
  "device": "mps",
  "method": "LoRA (no quantisation) on glossary Q&A from scripts/make_sft_dataset.py",
  "train_records": 104,
  "held_out_terms": [
    "harness",
    "pi",
    "tool-namespace",
    "mcp-config",
    "fetch-server",
    "lora-rank"
  ],
  "training_seconds": 2378,
  "group_sizes": {
    "trained, training wording": 6,
    "trained, new wording": 6,
    "trained, described without the term": 4,
    "in glossary, not trained": 6,
    "not in glossary": 4
  },
  "correct": {
    "base, closed book": {
      "trained, trai